# Capa Bronze

## Importando

In [0]:
%python
from pyspark.sql import SparkSession
from pyspark.sql.functions import col


## crm_cust_info

In [0]:
# -------------------------------
# Capa Bronze - CRM Customers
# -------------------------------

# 1 Importando el CSV al DataFrame
df_cust = spark.read.option("header", "true") \
                    .option("inferSchema", "true") \
                    .option("delimiter", ",") \
                    .csv("/Volumes/workspace/default/data-warehouse-databricks/crm_cust_info.csv")

# 2 Validando el schema y revisando data
df_cust.printSchema()
df_cust.show(5)

root
 |-- cst_id: integer (nullable = true)
 |-- cst_key: string (nullable = true)
 |-- cst_firstname: string (nullable = true)
 |-- cst_lastname: string (nullable = true)
 |-- cst_marital_status: string (nullable = true)
 |-- cst_gndr: string (nullable = true)
 |-- cst_create_date: date (nullable = true)

+------+----------+-------------+------------+------------------+--------+---------------+
|cst_id|   cst_key|cst_firstname|cst_lastname|cst_marital_status|cst_gndr|cst_create_date|
+------+----------+-------------+------------+------------------+--------+---------------+
| 11000|AW00011000|          Jon|       Yang |                 M|       M|     2025-10-06|
| 11001|AW00011001|       Eugene|     Huang  |                 S|       M|     2025-10-06|
| 11002|AW00011002|        Ruben|      Torres|                 M|       M|     2025-10-06|
| 11003|AW00011003|      Christy|         Zhu|                 S|       F|     2025-10-06|
| 11004|AW00011004|    Elizabeth|     Johnson|         

In [0]:
# 3 Sobreescribir tabla Delta en Bronze
df_cust.write.format("delta") \
       .mode("overwrite") \
       .saveAsTable("bronze.crm_cust_info")

# 4 Cargar la tabla Delta de Bronze en un DataFrame para confirmar la creación
df_cust = spark.table("bronze.crm_cust_info")

df_cust.show(5)
df_cust.printSchema()

+------+----------+-------------+------------+------------------+--------+---------------+
|cst_id|   cst_key|cst_firstname|cst_lastname|cst_marital_status|cst_gndr|cst_create_date|
+------+----------+-------------+------------+------------------+--------+---------------+
| 11000|AW00011000|          Jon|       Yang |                 M|       M|     2025-10-06|
| 11001|AW00011001|       Eugene|     Huang  |                 S|       M|     2025-10-06|
| 11002|AW00011002|        Ruben|      Torres|                 M|       M|     2025-10-06|
| 11003|AW00011003|      Christy|         Zhu|                 S|       F|     2025-10-06|
| 11004|AW00011004|    Elizabeth|     Johnson|                 S|       F|     2025-10-06|
+------+----------+-------------+------------+------------------+--------+---------------+
only showing top 5 rows
root
 |-- cst_id: integer (nullable = true)
 |-- cst_key: string (nullable = true)
 |-- cst_firstname: string (nullable = true)
 |-- cst_lastname: string (nu

## crm_prd_info

In [0]:

# 1 Leer CSV en un DataFrame
df_prd = spark.read.option("header", "true") \
                   .option("inferSchema", "true") \
                   .option("delimiter", ",") \
                   .csv("/Volumes/workspace/default/data-warehouse-databricks/crm_prd_info.csv")

# 2 Validar esquema y previsualizar datos
df_prd.printSchema()
df_prd.show(5)

root
 |-- prd_id: integer (nullable = true)
 |-- prd_key: string (nullable = true)
 |-- prd_nm: string (nullable = true)
 |-- prd_cost: integer (nullable = true)
 |-- prd_line: string (nullable = true)
 |-- prd_start_dt: date (nullable = true)
 |-- prd_end_dt: date (nullable = true)

+------+----------------+--------------------+--------+--------+------------+----------+
|prd_id|         prd_key|              prd_nm|prd_cost|prd_line|prd_start_dt|prd_end_dt|
+------+----------------+--------------------+--------+--------+------------+----------+
|   210|CO-RF-FR-R92B-58|HL Road Frame - B...|    NULL|      R |  2003-07-01|      NULL|
|   211|CO-RF-FR-R92R-58|HL Road Frame - R...|    NULL|      R |  2003-07-01|      NULL|
|   212| AC-HE-HL-U509-R|Sport-100 Helmet-...|      12|      S |  2011-07-01|2007-12-28|
|   213| AC-HE-HL-U509-R|Sport-100 Helmet-...|      14|      S |  2012-07-01|2008-12-27|
|   214| AC-HE-HL-U509-R|Sport-100 Helmet-...|      13|      S |  2013-07-01|      NULL|
+--

In [0]:
# 3 Sobreescribir tabla Delta en Bronze

df_prd_fixed = df_prd.withColumn("prd_start_dt", col("prd_start_dt").cast("timestamp")) \
                     .withColumn("prd_end_dt", col("prd_end_dt").cast("timestamp"))

df_prd_fixed.write.format("delta") \
       .mode("overwrite") \
       .saveAsTable("bronze.crm_prd_info")

# 4 Cargar la tabla Delta de Bronze para confirmar
df_prd = spark.table("bronze.crm_prd_info")
df_prd.show(5)
df_prd.printSchema()
     

+------+----------------+--------------------+--------+--------+-------------------+-------------------+
|prd_id|         prd_key|              prd_nm|prd_cost|prd_line|       prd_start_dt|         prd_end_dt|
+------+----------------+--------------------+--------+--------+-------------------+-------------------+
|   210|CO-RF-FR-R92B-58|HL Road Frame - B...|    NULL|      R |2003-07-01 00:00:00|               NULL|
|   211|CO-RF-FR-R92R-58|HL Road Frame - R...|    NULL|      R |2003-07-01 00:00:00|               NULL|
|   212| AC-HE-HL-U509-R|Sport-100 Helmet-...|      12|      S |2011-07-01 00:00:00|2007-12-28 00:00:00|
|   213| AC-HE-HL-U509-R|Sport-100 Helmet-...|      14|      S |2012-07-01 00:00:00|2008-12-27 00:00:00|
|   214| AC-HE-HL-U509-R|Sport-100 Helmet-...|      13|      S |2013-07-01 00:00:00|               NULL|
+------+----------------+--------------------+--------+--------+-------------------+-------------------+
only showing top 5 rows
root
 |-- prd_id: integer (null

## crm_sales_details

In [0]:
# 1 Leer CSV en un DataFrame
df_sales = spark.read.option("header", "true") \
                     .option("inferSchema", "true") \
                     .option("delimiter", ",") \
                     .csv("/Volumes/workspace/default/data-warehouse-databricks/crm_sales_details.csv")

# 2 Validar esquema y previsualizar datos
df_sales.printSchema()
df_sales.show(5)

root
 |-- sls_ord_num: string (nullable = true)
 |-- sls_prd_key: string (nullable = true)
 |-- sls_cust_id: integer (nullable = true)
 |-- sls_order_dt: integer (nullable = true)
 |-- sls_ship_dt: integer (nullable = true)
 |-- sls_due_dt: integer (nullable = true)
 |-- sls_sales: integer (nullable = true)
 |-- sls_quantity: integer (nullable = true)
 |-- sls_price: integer (nullable = true)

+-----------+-----------+-----------+------------+-----------+----------+---------+------------+---------+
|sls_ord_num|sls_prd_key|sls_cust_id|sls_order_dt|sls_ship_dt|sls_due_dt|sls_sales|sls_quantity|sls_price|
+-----------+-----------+-----------+------------+-----------+----------+---------+------------+---------+
|    SO43697| BK-R93R-62|      21768|    20101229|   20110105|  20110110|     3578|           1|     3578|
|    SO43698| BK-M82S-44|      28389|    20101229|   20110105|  20110110|     3400|           1|     3400|
|    SO43699| BK-M82S-44|      25863|    20101229|   20110105|  2011

In [0]:
# 3 Sobreescribir tabla Delta en Bronze
df_sales.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable("bronze.crm_sales_details")

# 4 Cargar la tabla Delta de Bronze para confirmar
df_sales = spark.table("bronze.crm_sales_details")
df_sales.show(5)
df_sales.printSchema()
     

+-----------+-----------+-----------+------------+-----------+----------+---------+------------+---------+
|sls_ord_num|sls_prd_key|sls_cust_id|sls_order_dt|sls_ship_dt|sls_due_dt|sls_sales|sls_quantity|sls_price|
+-----------+-----------+-----------+------------+-----------+----------+---------+------------+---------+
|    SO43697| BK-R93R-62|      21768|    20101229|   20110105|  20110110|     3578|           1|     3578|
|    SO43698| BK-M82S-44|      28389|    20101229|   20110105|  20110110|     3400|           1|     3400|
|    SO43699| BK-M82S-44|      25863|    20101229|   20110105|  20110110|     3400|           1|     3400|
|    SO43700| BK-R50B-62|      14501|    20101229|   20110105|  20110110|      699|           1|      699|
|    SO43701| BK-M82S-44|      11003|    20101229|   20110105|  20110110|     3400|           1|     3400|
+-----------+-----------+-----------+------------+-----------+----------+---------+------------+---------+
only showing top 5 rows
root
 |-- sls

## erp_cust_az12

In [0]:
# 1 Leer CSV en un DataFrame
df_erp_cust = spark.read.option("header", "true") \
                        .option("inferSchema", "true") \
                        .option("delimiter", ",") \
                        .csv("/Volumes/workspace/default/data-warehouse-databricks/erp_CUST_AZ12.csv")

# 2 Validar esquema y previsualizar
df_erp_cust.printSchema()
df_erp_cust.show(5)

root
 |-- CID: string (nullable = true)
 |-- BDATE: date (nullable = true)
 |-- GEN: string (nullable = true)

+-------------+----------+------+
|          CID|     BDATE|   GEN|
+-------------+----------+------+
|NASAW00011000|1971-10-06|  Male|
|NASAW00011001|1976-05-10|  Male|
|NASAW00011002|1971-02-09|  Male|
|NASAW00011003|1973-08-14|Female|
|NASAW00011004|1979-08-05|Female|
+-------------+----------+------+
only showing top 5 rows


In [0]:
# 3 Sobreescribir tabla Delta en Bronze
df_erp_cust.write.format("delta") \
           .mode("overwrite") \
           .saveAsTable("bronze.erp_cust_az12")

# 4 Cargar la tabla Delta de Bronze para confirmar
df_erp_cust = spark.table("bronze.erp_cust_az12")
df_erp_cust.show(5)
df_erp_cust.printSchema()

+-------------+----------+------+
|          cid|     bdate|   gen|
+-------------+----------+------+
|NASAW00011000|1971-10-06|  Male|
|NASAW00011001|1976-05-10|  Male|
|NASAW00011002|1971-02-09|  Male|
|NASAW00011003|1973-08-14|Female|
|NASAW00011004|1979-08-05|Female|
+-------------+----------+------+
only showing top 5 rows
root
 |-- cid: string (nullable = true)
 |-- bdate: date (nullable = true)
 |-- gen: string (nullable = true)



## erp_loc_a101

In [0]:
# 1 Leer CSV en un DataFrame
df_erp_loc = spark.read.option("header", "true") \
                       .option("inferSchema", "true") \
                       .option("delimiter", ",") \
                       .csv("/Volumes/workspace/default/data-warehouse-databricks/erp_LOC_A101.csv")

# 2 Validar esquema y previsualizar
df_erp_loc.printSchema()
df_erp_loc.show(5)

root
 |-- CID: string (nullable = true)
 |-- CNTRY: string (nullable = true)

+-----------+---------+
|        CID|    CNTRY|
+-----------+---------+
|AW-00011000|Australia|
|AW-00011001|Australia|
|AW-00011002|Australia|
|AW-00011003|Australia|
|AW-00011004|Australia|
+-----------+---------+
only showing top 5 rows


In [0]:
# 3 Sobreescribir tabla Delta en Bronze
df_erp_loc.write.format("delta") \
          .mode("overwrite") \
          .saveAsTable("bronze.erp_loc_a101")

# 4 Cargar la tabla Delta de Bronze para confirmar
df_erp_loc = spark.table("bronze.erp_loc_a101")
df_erp_loc.show(5)
df_erp_loc.printSchema()

+-----------+---------+
|        cid|    cntry|
+-----------+---------+
|AW-00011000|Australia|
|AW-00011001|Australia|
|AW-00011002|Australia|
|AW-00011003|Australia|
|AW-00011004|Australia|
+-----------+---------+
only showing top 5 rows
root
 |-- cid: string (nullable = true)
 |-- cntry: string (nullable = true)



## erp_px_cat_g1v2

In [0]:
# 1 Leer CSV en un DataFrame
df_erp_prd = spark.read.option("header", "true") \
                       .option("inferSchema", "true") \
                       .option("delimiter", ",") \
                       .csv("/Volumes/workspace/default/data-warehouse-databricks/erp_PX_CAT_G1V2.csv")

# 2 Validar esquema y previsualizar
df_erp_prd.printSchema()
df_erp_prd.show(5)
     

root
 |-- ID: string (nullable = true)
 |-- CAT: string (nullable = true)
 |-- SUBCAT: string (nullable = true)
 |-- MAINTENANCE: string (nullable = true)

+-----+-----------+-----------------+-----------+
|   ID|        CAT|           SUBCAT|MAINTENANCE|
+-----+-----------+-----------------+-----------+
|AC_BR|Accessories|       Bike Racks|        Yes|
|AC_BS|Accessories|      Bike Stands|         No|
|AC_BC|Accessories|Bottles and Cages|         No|
|AC_CL|Accessories|         Cleaners|        Yes|
|AC_FE|Accessories|          Fenders|         No|
+-----+-----------+-----------------+-----------+
only showing top 5 rows


In [0]:
# 3 Sobreescribir tabla Delta en Bronze
df_erp_prd.write.format("delta") \
           .mode("overwrite") \
           .saveAsTable("bronze.erp_px_cat_g1v2")

# 4 Cargar la tabla Delta de Bronze para confirmar
df_erp_prd = spark.table("bronze.erp_px_cat_g1v2")
df_erp_prd.show(5)
df_erp_prd.printSchema()

+-----+-----------+-----------------+-----------+
|   id|        cat|           subcat|maintenance|
+-----+-----------+-----------------+-----------+
|AC_BR|Accessories|       Bike Racks|        Yes|
|AC_BS|Accessories|      Bike Stands|         No|
|AC_BC|Accessories|Bottles and Cages|         No|
|AC_CL|Accessories|         Cleaners|        Yes|
|AC_FE|Accessories|          Fenders|         No|
+-----+-----------+-----------------+-----------+
only showing top 5 rows
root
 |-- id: string (nullable = true)
 |-- cat: string (nullable = true)
 |-- subcat: string (nullable = true)
 |-- maintenance: string (nullable = true)

